In [1]:
# Dependencies managed via FinOps_Environment
print("Environment: FinOps_Environment")
print("All dependencies loaded.")

StatementMeta(, d99bae5d-d10d-4294-9e97-5a5965262a65, 4, Finished, Available, Finished, False)

Environment: FinOps_Environment
All dependencies loaded.


In [2]:
# Watermark management functions
from datetime import datetime

WATERMARK_BASE_PATH = "Files/watermarks"
DEFAULT_WATERMARK   = "1900-01-01 00:00:00"

def get_wm_last_update(table_name: str) -> str:
    """Read the last watermark timestamp for a given table."""
    path = f"{WATERMARK_BASE_PATH}/{table_name}.txt"
    try:
        value = mssparkutils.fs.head(path, 1024).strip()
        print(f"  [WM] {table_name} → last loaded: {value}")
        return value
    except:
        print(f"  [WM] {table_name} → no watermark found, using default: {DEFAULT_WATERMARK}")
        return DEFAULT_WATERMARK

def update_wm(table_name: str, latest_update: str) -> None:
    """Write the new watermark timestamp for a given table."""
    path = f"{WATERMARK_BASE_PATH}/{table_name}.txt"
    mssparkutils.fs.put(path, latest_update.strip(), overwrite=True)
    print(f"  [WM] {table_name} → watermark updated to: {latest_update}")

StatementMeta(, d99bae5d-d10d-4294-9e97-5a5965262a65, 5, Finished, Available, Finished, False)

In [3]:
# Load configuration from Lakehouse config file
import json

CONFIG_PATH = "/lakehouse/default/Files/config/finops_config.json"

with open(CONFIG_PATH, "r") as f:
    config = json.load(f)

# Extract configs
MYSQL_CONFIG  = config["mysql"]
PG_CONFIG     = config["postgresql"]
GS_CONFIG     = config["google_sheets"]
BRONZE_BASE   = config["paths"]["bronze_base"]
WATERMARK_BASE = config["paths"]["watermark_base"]

print("Configuration loaded successfully from Lakehouse.")
print(f"  MySQL host:      {MYSQL_CONFIG['host']}")
print(f"  PostgreSQL host: {PG_CONFIG['host']}")
print(f"  Sheet name:      {GS_CONFIG['sheet_name']}")
print(f"  Bronze path:     {BRONZE_BASE}")
print(f"  Watermark path:  {WATERMARK_BASE}")

StatementMeta(, d99bae5d-d10d-4294-9e97-5a5965262a65, 6, Finished, Available, Finished, False)

Configuration loaded successfully from Lakehouse.
  MySQL host:      finops-mysql-ndianabernard2-92be.g.aivencloud.com
  PostgreSQL host: aws-0-eu-west-1.pooler.supabase.com
  Sheet name:      FinOps_SupportTickets
  Bronze path:     Files/bronze
  Watermark path:  Files/watermarks


In [4]:
# MySQL Bronze Ingestion
import mysql.connector
import pandas as pd
from datetime import datetime

def ingest_mysql_table(table_name: str, watermark_col: str):
    print(f"\n── MySQL: {table_name}")
    
    # Read watermark
    last_wm = get_wm_last_update(table_name)
    
    # Connect to MySQL
    conn = mysql.connector.connect(**MYSQL_CONFIG)
    cursor = conn.cursor(dictionary=True)
    
    # Query only new/changed rows
    query = f"""
        SELECT * FROM {table_name}
        WHERE {watermark_col} > '{last_wm}'
        ORDER BY {watermark_col} ASC
    """
    cursor.execute(query)
    rows = cursor.fetchall()
    cursor.close()
    conn.close()
    
    if not rows:
        print(f"  No new rows found since {last_wm}. Skipping.")
        return
    
    print(f"  Fetched {len(rows):,} rows from MySQL")
    
    # Convert to Spark DataFrame
    df = spark.createDataFrame(pd.DataFrame(rows))
    
    # Write to Bronze as Parquet
    bronze_path = f"{BRONZE_BASE}/mysql/{table_name}"
    df.write.mode("append").parquet(f"Files/{bronze_path.replace('Files/','')}")
    
    print(f"  Written to: {bronze_path}")
    
    # Update watermark with max value of watermark column
    new_wm = str(pd.DataFrame(rows)[watermark_col].max())
    update_wm(table_name, new_wm)

# Run for all MySQL tables
print("Starting MySQL Bronze Ingestion...")
print(f"Run time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

ingest_mysql_table("customers",     "updated_at")
ingest_mysql_table("subscriptions", "updated_at")
ingest_mysql_table("fx_rates",      "rate_date")

print("\nMySQL Bronze Ingestion complete.")

StatementMeta(, d99bae5d-d10d-4294-9e97-5a5965262a65, 7, Finished, Available, Finished, False)

Starting MySQL Bronze Ingestion...
Run time: 2026-06-20 07:09:59

── MySQL: customers
  [WM] customers → last loaded: 2026-06-16 12:38:55
  Fetched 40 rows from MySQL
  Written to: Files/bronze/mysql/customers
  [WM] customers → watermark updated to: 2026-06-19 11:04:40

── MySQL: subscriptions
  [WM] subscriptions → last loaded: 2026-06-16 12:39:48
  Fetched 30 rows from MySQL
  Written to: Files/bronze/mysql/subscriptions
  [WM] subscriptions → watermark updated to: 2026-06-19 11:04:36

── MySQL: fx_rates
  [WM] fx_rates → last loaded: 2026-06-16
  Fetched 1 rows from MySQL
  Written to: Files/bronze/mysql/fx_rates
  [WM] fx_rates → watermark updated to: 2026-06-19

MySQL Bronze Ingestion complete.


In [5]:
# PostgreSQL Bronze Ingestion (towers, network_events, vendors)
import psycopg2
import psycopg2.extras
import pandas as pd
from pyspark.sql.functions import col, lit
from pyspark.sql.types import StringType

def cast_void_columns(df):
    """Cast any VOID/NullType columns to StringType to avoid Parquet write errors."""
    from pyspark.sql.types import NullType
    for field in df.schema.fields:
        if isinstance(field.dataType, NullType):
            print(f"  [CAST] Column '{field.name}' is VOID → casting to String")
            df = df.withColumn(field.name, col(field.name).cast(StringType()))
    return df

def ingest_pg_table(table_name: str, watermark_col: str):
    print(f"\n── PostgreSQL: {table_name}")
    
    last_wm = get_wm_last_update(table_name)
    
    conn = psycopg2.connect(**PG_CONFIG)
    cursor = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    
    query = f"""
        SELECT * FROM {table_name}
        WHERE {watermark_col} > '{last_wm}'
        ORDER BY {watermark_col} ASC
    """
    cursor.execute(query)
    rows = cursor.fetchall()
    cursor.close()
    conn.close()
    
    if not rows:
        print(f"  No new rows found since {last_wm}. Skipping.")
        return
    
    print(f"  Fetched {len(rows):,} rows from PostgreSQL")
    
    df = spark.createDataFrame(pd.DataFrame([dict(r) for r in rows]))
    
    # Fix VOID columns before writing
    df = cast_void_columns(df)
    
    bronze_path = f"Files/bronze/postgres/{table_name}"
    df.write.mode("append").parquet(bronze_path)
    
    print(f"  Written to: {bronze_path}")
    
    new_wm = str(pd.DataFrame([dict(r) for r in rows])[watermark_col].max())
    update_wm(table_name, new_wm)

# Run for all PostgreSQL tables
print("Starting PostgreSQL Bronze Ingestion...")
print(f"Run time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

ingest_pg_table("towers",         "updated_at")
ingest_pg_table("network_events", "created_at")
ingest_pg_table("vendors",        "updated_at")

print("\nPostgreSQL Bronze Ingestion complete.")

StatementMeta(, d99bae5d-d10d-4294-9e97-5a5965262a65, 8, Finished, Available, Finished, False)

Starting PostgreSQL Bronze Ingestion...
Run time: 2026-06-20 07:10:11

── PostgreSQL: towers
  [WM] towers → last loaded: 2026-06-16 12:44:26.903923
  Fetched 5 rows from PostgreSQL
  Written to: Files/bronze/postgres/towers
  [WM] towers → watermark updated to: 2026-06-19 11:06:57.583729

── PostgreSQL: network_events
  [WM] network_events → last loaded: 2026-06-16 13:43:23.936508
  Fetched 40 rows from PostgreSQL
  [CAST] Column 'notes' is VOID → casting to String
  Written to: Files/bronze/postgres/network_events
  [WM] network_events → watermark updated to: 2026-06-19 12:06:53.108223

── PostgreSQL: vendors
  [WM] vendors → last loaded: 2026-06-16 12:53:42.485678
  Fetched 5 rows from PostgreSQL
  Written to: Files/bronze/postgres/vendors
  [WM] vendors → watermark updated to: 2026-06-19 11:06:55.699318

PostgreSQL Bronze Ingestion complete.


In [6]:
# Google Sheets Bronze Ingestion (support_tickets)
import gspread
from google.oauth2.service_account import Credentials
import pandas as pd

def ingest_google_sheet():
    print(f"\n── Google Sheets: support_tickets")
    
    last_wm = get_wm_last_update("support_tickets")
    
    # Authenticate
    SCOPES = [
        'https://www.googleapis.com/auth/spreadsheets',
        'https://www.googleapis.com/auth/drive'
    ]
    creds  = Credentials.from_service_account_file(
        GS_CONFIG["credentials_path"], scopes=SCOPES
    )
    client = gspread.authorize(creds)
    
    # Open sheet and get all records
    sheet  = client.open(GS_CONFIG["sheet_name"]).sheet1
    data   = sheet.get_all_records()
    
    if not data:
        print("  No data found in Google Sheet. Skipping.")
        return
    
    df = pd.DataFrame(data)
    print(f"  Fetched {len(df):,} total rows from Google Sheets")
    
    # Apply watermark filter on CreatedDate
    df["CreatedDate"] = pd.to_datetime(df["CreatedDate"], errors="coerce")
    df_filtered = df[df["CreatedDate"] > pd.to_datetime(last_wm)]
    
    if df_filtered.empty:
        print(f"  No new rows since {last_wm}. Skipping.")
        return
    
    print(f"  Filtered to {len(df_filtered):,} new rows after watermark")
    
    # Convert to Spark DataFrame
    df_filtered["CreatedDate"]  = df_filtered["CreatedDate"].astype(str)
    df_filtered["ResolvedDate"] = df_filtered["ResolvedDate"].astype(str)
    
    spark_df = spark.createDataFrame(df_filtered)
    
    bronze_path = "Files/bronze/gsheets/support_tickets"
    spark_df.write.mode("append").parquet(bronze_path)
    
    print(f"  Written to: {bronze_path}")
    
    # Update watermark
    new_wm = str(df_filtered["CreatedDate"].max())
    update_wm("support_tickets", new_wm)

# Run
print("Starting Google Sheets Bronze Ingestion...")
print(f"Run time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

ingest_google_sheet()

print("\nGoogle Sheets Bronze Ingestion complete.")

StatementMeta(, d99bae5d-d10d-4294-9e97-5a5965262a65, 9, Finished, Available, Finished, False)

Starting Google Sheets Bronze Ingestion...
Run time: 2026-06-20 07:10:19

── Google Sheets: support_tickets
  [WM] support_tickets → last loaded: 2026-06-16 14:45:26
  Written to: Files/bronze/gsheets/support_tickets
  [WM] support_tickets → watermark updated to: 2026-06-20 08:05:32

Google Sheets Bronze Ingestion complete.


/tmp/ipykernel_8442/13797804.py:43: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered["CreatedDate"]  = df_filtered["CreatedDate"].astype(str)
/tmp/ipykernel_8442/13797804.py:44: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered["ResolvedDate"] = df_filtered["ResolvedDate"].astype(str)
/opt/spark/python/lib/pyspark.zip/pyspark/sql/pandas/conversion.py:351: UserWarning: createDataFrame attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, fa